In [ ]:
import statbotics
import json
from datetime import datetime
from pathlib import Path

sb = statbotics.Statbotics()
print("Statbotics API client initialized")

def extract_event_epa_data(event_key: str, output_file: str = None):
    """Extract post-match EPA data for all teams in an event"""

    matches = sb.get_matches(event=event_key, limit=250)
    output_path = Path(output_file) if output_file else None

    saved_matches = []
    existing_keys = set()
    if output_path and output_path.exists():
        try:
            with output_path.open() as f:
                previous = json.load(f)
        except (json.JSONDecodeError, OSError):
            previous = {}
        if previous.get("event") == event_key:
            saved_matches = list(previous.get("matches", []))
            existing_keys = {m.get("match_key") for m in saved_matches if m.get("match_key")}

    event_data = {
        "event": event_key,
        "extracted_at": datetime.now().isoformat(),
        "matches": list(saved_matches)
    }

    start_index = 0
    if saved_matches:
        last_key = saved_matches[-1].get("match_key")
        for idx, match in enumerate(matches):
            if match.get("key") == last_key:
                start_index = idx + 1
                break

    def write_snapshot():
        if not output_path:
            return
        event_data["extracted_at"] = datetime.now().isoformat()
        with output_path.open("w") as f:
            json.dump(event_data, f, indent=2)
        print(f"Snapshot saved with {len(event_data['matches'])} matches")

    processed_since_save = 0

    for match in matches[start_index:]:
        match_key = match.get("key")
        if match_key in existing_keys:
            continue

        match_data = {
            "match_key": match_key,
            "time": match.get("time"),
            "status": match.get("status"),
            "teams": []
        }

        red_teams = match["alliances"]["red"]["team_keys"]
        blue_teams = match["alliances"]["blue"]["team_keys"]

        for team in red_teams + blue_teams:
            alliance = "red" if team in red_teams else "blue"
            team_match = sb.get_team_match(team=team, match=match_key)
            epa_entry = None

            if team_match:
                epa_data = team_match.get("epa")
                if epa_data:
                    pre = epa_data.get("total_points")
                    post = epa_data.get("post")
                    epa_change = None
                    if isinstance(pre, (int, float)) and isinstance(post, (int, float)):
                        epa_change = post - pre

                    epa_entry = {
                        "pre_match_total": pre,
                        "post_match_total": post,
                        "epa_change": epa_change,
                        "breakdown": epa_data.get("breakdown")
                    }

            match_data["teams"].append({
                "team": team,
                "alliance": alliance,
                "epa": epa_entry
            })

        if all(team["epa"] is None for team in match_data["teams"]):
            continue

        event_data["matches"].append(match_data)
        existing_keys.add(match_key)
        processed_since_save += 1

        if processed_since_save >= 5:
            write_snapshot()
            processed_since_save = 0

    if processed_since_save > 0:
        write_snapshot()

    return event_data


# Example usage
event_data = extract_event_epa_data("2026orore", "Cais_epa_data(1).json")
print(f"Extracted EPA data for {len(event_data['matches'])} matches")


Statbotics API client initialized
Snapshot saved with 5 matches
Snapshot saved with 10 matches
Snapshot saved with 15 matches
Snapshot saved with 20 matches
Snapshot saved with 25 matches
Snapshot saved with 30 matches
Snapshot saved with 35 matches
Snapshot saved with 40 matches
Snapshot saved with 45 matches
Snapshot saved with 50 matches
Snapshot saved with 55 matches
Snapshot saved with 60 matches
Snapshot saved with 65 matches
Snapshot saved with 70 matches
